# 🧠 Deep Autoencoder with ResNet Backbone
## Unsupervised Anomaly Detection — Multi-Class Dataset

This notebook trains a **Deep Convolutional Autoencoder** with a **ResNet-18 encoder backbone** on a user-provided dataset.

**Expected folder structure:**
```
dataset_root/
    class_A/
        img001.png
        img002.jpg
        ...
    class_B/
        img001.png
        ...
    class_C/
        ...
```

**Features:**
- ResNet-18 pretrained encoder (ImageNet weights)
- Multi-scale decoder with skip connections
- Per-class training support
- Reconstruction error heatmaps
- Latent space visualization (t-SNE)

In [ ]:
# ============================================================
# 1. INSTALL & IMPORTS
# ============================================================
!pip install pytorch-msssim timm -q

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models
from pytorch_msssim import SSIM
from sklearn.manifold import TSNE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# 2. CONFIGURATION  ← Set your dataset path here
# ============================================================

# Path to your dataset root (subfolders = classes)
DATASET_PATH  = '/content/my_dataset'   # <-- CHANGE THIS

IMG_SIZE      = 224       # ResNet expects 224x224
BATCH_SIZE    = 32
EPOCHS        = 80
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
VAL_SPLIT     = 0.15      # fraction for validation
TRAIN_ALL     = True      # True = train on all classes together; False = one model per class
PRETRAINED    = True      # Use ImageNet pretrained ResNet encoder
LATENT_DIM    = 256       # bottleneck feature dimension
SAVE_DIR      = '/content/checkpoints'

os.makedirs(SAVE_DIR, exist_ok=True)
print('Config OK')

In [ ]:
# ============================================================
# 3. DATASET — Auto-discovers classes from subfolders
# ============================================================

class FolderDataset(Dataset):
    """Loads images from root/class_name/*.ext structure."""

    EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

    def __init__(self, root, img_size=224, augment=False, class_filter=None):
        self.samples   = []  # (filepath, class_idx)
        self.class_names = []
        self.class_to_idx = {}

        root = os.path.expanduser(root)
        assert os.path.isdir(root), f'Dataset root not found: {root}'

        classes = sorted([d for d in os.listdir(root)
                          if os.path.isdir(os.path.join(root, d))])
        assert len(classes) > 0, 'No subfolders (classes) found in dataset root!'

        if class_filter:
            classes = [c for c in classes if c in class_filter]

        self.class_names  = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}

        for cls in classes:
            cls_dir = os.path.join(root, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(self.EXTENSIONS):
                    self.samples.append((os.path.join(cls_dir, fname),
                                         self.class_to_idx[cls]))

        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]
        aug_transforms = [
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.RandomRotation(15),
        ] if augment else []

        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            *aug_transforms,
            T.ToTensor(),
            T.Normalize(mean, std)
        ])
        self.inv_norm = T.Normalize(
            mean=[-m/s for m, s in zip(mean, std)],
            std=[1/s for s in std]
        )

        print(f'Dataset: {len(self.samples)} images | {len(classes)} classes: {classes}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


full_dataset = FolderDataset(DATASET_PATH, img_size=IMG_SIZE, augment=True)
NUM_CLASSES  = len(full_dataset.class_names)
print(f'Discovered {NUM_CLASSES} classes: {full_dataset.class_names}')

# Train/val split
n_val   = int(len(full_dataset) * VAL_SPLIT)
n_train = len(full_dataset) - n_val
train_set, val_set = random_split(full_dataset, [n_train, n_val],
                                   generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {n_train} | Val: {n_val}')

In [ ]:
# ============================================================
# 4. RESNET ENCODER + DEEP DECODER ARCHITECTURE
# ============================================================

class ResNetEncoder(nn.Module):
    """ResNet-18 encoder that extracts multi-scale feature maps."""

    def __init__(self, pretrained=True, latent_dim=256):
        super().__init__()
        resnet = models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)

        # Feature extraction layers (no FC, no avgpool)
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 112x112, 64ch
        self.pool   = resnet.maxpool                                          # 56x56
        self.layer1 = resnet.layer1   # 56x56,  64ch
        self.layer2 = resnet.layer2   # 28x28, 128ch
        self.layer3 = resnet.layer3   # 14x14, 256ch
        self.layer4 = resnet.layer4   # 7x7,   512ch

        # Bottleneck projection
        self.bottleneck = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, latent_dim),
            nn.ReLU(inplace=True)
        )
        self.bottleneck_spatial = nn.Sequential(
            nn.Conv2d(512, latent_dim, 1, bias=False),
            nn.BatchNorm2d(latent_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        s0 = self.layer0(x)      # 112x112, 64
        p  = self.pool(s0)       # 56x56
        s1 = self.layer1(p)      # 56x56, 64
        s2 = self.layer2(s1)     # 28x28, 128
        s3 = self.layer3(s2)     # 14x14, 256
        s4 = self.layer4(s3)     # 7x7,   512

        z_spatial = self.bottleneck_spatial(s4)  # 7x7, latent_dim
        z_vector  = self.bottleneck(s4)           # latent_dim
        return z_spatial, z_vector, (s0, s1, s2, s3)


class UpBlock(nn.Module):
    """Upsample + Conv block with optional skip connection."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            # Pad if needed
            diffH = skip.size(2) - x.size(2)
            diffW = skip.size(3) - x.size(3)
            x = F.pad(x, [diffW//2, diffW-diffW//2, diffH//2, diffH-diffH//2])
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResNetDecoder(nn.Module):
    """U-Net style decoder that reconstructs to 224x224x3."""

    def __init__(self, latent_dim=256):
        super().__init__()
        # From 7x7 latent_dim → 224x224 RGB
        self.up0 = UpBlock(latent_dim, 256, 256)   # 14x14
        self.up1 = UpBlock(256,        128, 128)   # 28x28
        self.up2 = UpBlock(128,         64, 128)   # 56x56  (skip from layer1)
        self.up3 = UpBlock(128,         64,  64)   # 112x112 (skip from layer0)
        self.up4 = UpBlock(64,           0,  32)   # 224x224
        self.final = nn.Sequential(
            nn.Conv2d(32, 3, 1),
            nn.Sigmoid()
        )

    def forward(self, z, skips):
        s0, s1, s2, s3 = skips
        x = self.up0(z, s3)   # 14x14
        x = self.up1(x, s2)   # 28x28
        x = self.up2(x, s1)   # 56x56
        x = self.up3(x, s0)   # 112x112
        x = self.up4(x)       # 224x224
        return self.final(x)


class ResNetAutoencoder(nn.Module):
    def __init__(self, pretrained=True, latent_dim=256):
        super().__init__()
        self.encoder = ResNetEncoder(pretrained=pretrained, latent_dim=latent_dim)
        self.decoder = ResNetDecoder(latent_dim=latent_dim)

    def forward(self, x):
        z_spatial, z_vector, skips = self.encoder(x)
        recon = self.decoder(z_spatial, skips)
        return recon, z_vector


model = ResNetAutoencoder(pretrained=PRETRAINED, latent_dim=LATENT_DIM).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

# Shape sanity check
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    out, z = model(dummy)
    print(f'Input: {dummy.shape} → Latent: {z.shape} → Output: {out.shape}')

In [ ]:
# ============================================================
# 5. COMBINED LOSS (MSE + SSIM + Perceptual)
# ============================================================

class CombinedReconLoss(nn.Module):
    def __init__(self, alpha_ssim=0.5, alpha_mse=0.3, alpha_l1=0.2):
        super().__init__()
        self.ssim_fn    = SSIM(data_range=1.0, size_average=True, channel=3)
        self.alpha_ssim = alpha_ssim
        self.alpha_mse  = alpha_mse
        self.alpha_l1   = alpha_l1

    def forward(self, pred, target):
        # Denormalize to [0,1] for SSIM
        def denorm(t):
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(t.device)
            std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(t.device)
            return torch.clamp(t * std + mean, 0, 1)

        # pred is already sigmoid [0,1], target is normalized
        target_01 = denorm(target)
        ssim_l = 1 - self.ssim_fn(pred, target_01)
        mse_l  = F.mse_loss(pred, target_01)
        l1_l   = F.l1_loss(pred, target_01)
        return self.alpha_ssim * ssim_l + self.alpha_mse * mse_l + self.alpha_l1 * l1_l


criterion = CombinedReconLoss(alpha_ssim=0.5, alpha_mse=0.3, alpha_l1=0.2)

# Differential LR: lower for pretrained encoder
optimizer = optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': LR * 0.1},
    {'params': model.decoder.parameters(), 'lr': LR},
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[LR*0.1, LR],
    steps_per_epoch=len(train_loader), epochs=EPOCHS
)

print('Optimizer ready (differential LR: encoder=1e-4, decoder=1e-3)')

In [ ]:
# ============================================================
# 6. TRAINING LOOP
# ============================================================

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
checkpoint_path = os.path.join(SAVE_DIR, 'best_resnet_ae.pth')

for epoch in range(1, EPOCHS + 1):
    # ---- TRAIN ----
    model.train()
    train_loss = 0.0
    for imgs, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        recon, _ = model(imgs)
        loss = criterion(recon, imgs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # ---- VALIDATE ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            recon, _ = model(imgs)
            val_loss += criterion(recon, imgs).item()

    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
            'class_names': full_dataset.class_names,
        }, checkpoint_path)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d} | Train: {avg_train:.5f} | Val: {avg_val:.5f} | Best: {best_val_loss:.5f}')

print(f'\n✅ Training complete. Best val loss: {best_val_loss:.5f}')

In [ ]:
# ============================================================
# 7. TRAINING CURVES
# ============================================================

plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss', color='steelblue')
plt.plot(history['val_loss'],   label='Val Loss',   color='coral')
plt.xlabel('Epoch')
plt.ylabel('Combined Loss (SSIM+MSE+L1)')
plt.title('ResNet Autoencoder Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('resnet_ae_training.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 8. RECONSTRUCTION VISUALIZATION
# ============================================================

# Load best checkpoint
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]} (val loss: {ckpt["val_loss"]:.5f})')

INV_NORM = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

def show_reconstructions(model, loader, n=8, title='Reconstructions'):
    imgs_list, recon_list, diff_list, labels_list = [], [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            recon, _ = model(imgs)
            # Denorm originals to [0,1]
            orig_01 = torch.clamp(torch.stack([INV_NORM(img) for img in imgs.cpu()]), 0, 1)
            diff = torch.abs(orig_01 - recon.cpu()).mean(dim=1)  # [B,H,W]
            imgs_list.extend(orig_01[:n])
            recon_list.extend(recon.cpu()[:n])
            diff_list.extend(diff[:n])
            labels_list.extend(labels[:n].tolist())
            if len(imgs_list) >= n: break

    fig, axes = plt.subplots(3, n, figsize=(n * 2.5, 8))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    row_labels = ['Input', 'Reconstruction', 'Diff Map']

    for col in range(min(n, len(imgs_list))):
        cls_name = full_dataset.class_names[labels_list[col]]
        axes[0, col].imshow(imgs_list[col].permute(1,2,0).numpy())
        axes[0, col].set_title(cls_name, fontsize=8)
        axes[1, col].imshow(recon_list[col].permute(1,2,0).numpy())
        im = axes[2, col].imshow(diff_list[col].numpy(), cmap='hot', vmin=0, vmax=0.3)
        for row in range(3):
            axes[row, col].axis('off')

    for row, lbl in enumerate(row_labels):
        axes[row, 0].set_ylabel(lbl, fontsize=10, rotation=90)

    plt.tight_layout()
    plt.savefig('reconstructions.png', dpi=150)
    plt.show()


show_reconstructions(model, val_loader, n=8)

In [ ]:
# ============================================================
# 9. RECONSTRUCTION ERROR DISTRIBUTION PER CLASS
# ============================================================

class_errors = defaultdict(list)

model.eval()
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Computing per-class errors'):
        imgs = imgs.to(device)
        recon, _ = model(imgs)
        orig_01 = torch.clamp(torch.stack([INV_NORM(img) for img in imgs.cpu()]), 0, 1)
        errors  = (orig_01 - recon.cpu()).pow(2).mean(dim=(1,2,3))  # per-image MSE
        for err, lbl in zip(errors.tolist(), labels.tolist()):
            class_errors[full_dataset.class_names[lbl]].append(err)

# Plot
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES * 1.5), 5))
data = [class_errors[cls] for cls in full_dataset.class_names]
bp = ax.boxplot(data, labels=full_dataset.class_names, patch_artist=True)
colors = cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xlabel('Class')
ax.set_ylabel('MSE Reconstruction Error')
ax.set_title('Per-Class Reconstruction Error Distribution (Validation Set)')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('per_class_errors.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 10. LATENT SPACE VISUALIZATION (t-SNE)
# ============================================================

latent_vecs, label_list = [], []

model.eval()
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Extracting latents'):
        imgs = imgs.to(device)
        _, z = model(imgs)
        latent_vecs.append(z.cpu().numpy())
        label_list.extend(labels.numpy())

latent_vecs = np.concatenate(latent_vecs, axis=0)
label_arr   = np.array(label_list)

print(f'Running t-SNE on {latent_vecs.shape[0]} samples...')
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(latent_vecs)-1))
emb  = tsne.fit_transform(latent_vecs)

plt.figure(figsize=(10, 8))
colors = cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for i, cls in enumerate(full_dataset.class_names):
    mask = label_arr == i
    plt.scatter(emb[mask, 0], emb[mask, 1], c=[colors[i]], label=cls,
                alpha=0.7, s=30, edgecolors='none')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.title('t-SNE of Latent Space — ResNet Autoencoder')
plt.xlabel('t-SNE dim 1')
plt.ylabel('t-SNE dim 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('tsne_latent.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 11. ANOMALY SCORING UTILITY (for deployment / inference)
# ============================================================

def score_image(model, img_path, threshold=None):
    """Score a single image. Returns (score, heatmap, is_anomaly)."""
    transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    img  = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        recon, z = model(tensor)
        orig_01  = torch.clamp(INV_NORM(tensor.squeeze().cpu()), 0, 1)
        heatmap  = (orig_01 - recon.cpu().squeeze()).abs().mean(0).numpy()
        score    = float(heatmap.mean())

    is_anomaly = (score > threshold) if threshold else None

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_01.permute(1,2,0).numpy()); axes[0].set_title('Input')
    axes[1].imshow(recon.cpu().squeeze().permute(1,2,0).numpy()); axes[1].set_title('Reconstruction')
    im = axes[2].imshow(heatmap, cmap='hot'); axes[2].set_title(f'Error Map | score={score:.4f}')
    plt.colorbar(im, ax=axes[2])
    for ax in axes: ax.axis('off')
    plt.suptitle(f'Anomaly: {is_anomaly}' if is_anomaly is not None else f'Score: {score:.4f}')
    plt.tight_layout()
    plt.show()
    return score, heatmap, is_anomaly


# Example usage:
# score, heatmap, flag = score_image(model, '/path/to/image.jpg', threshold=0.05)
print('score_image() ready. Use: score_image(model, "path/to/img.jpg", threshold=0.05)')

## Summary

| Component | Details |
|-----------|--------|
| **Encoder** | ResNet-18 (ImageNet pretrained), layers 0–4 |
| **Decoder** | U-Net style with skip connections, bilinear upsampling |
| **Loss** | SSIM (50%) + MSE (30%) + L1 (20%) |
| **Optimizer** | AdamW, differential LR (encoder ×0.1) |
| **Scheduler** | OneCycleLR |
| **Input** | 224×224 normalized RGB |

**Notes:**
- The model trained on all classes together will learn a shared reconstruction manifold
- For anomaly detection, images outside this distribution will have high reconstruction error
- Use `score_image()` for inference on new images
- Set `TRAIN_ALL=False` and loop per class for class-specific thresholds